# TPU probe — EDL-CMSO build 2

Cheap reconnaissance before committing 20 h of TPU quota to a 50-round run. It answers
four questions that no amount of local testing can:

1. Does `machine_shape: "TpuV38"` actually resolve to a TPU v3-8, and how many cores does
   one process see? *(Build 1 lost a session to a silently coerced accelerator string —
   only the run's own output is evidence.)*
2. Is `torch_xla` present, at what version, on which runtime (PJRT or XRT)?
3. Does **bf16** work, and how fast? TPU has no useful fp16 path — and bf16 keeps fp32's
   exponent range, which is precisely what the GPU run lacked when it blew up at round 25.
4. **Does the real EDLCMSO model compile and take a training step on XLA?**

Expected cost: a few minutes of the 20 h TPU quota.


In [ ]:
# ── Cell 1 · what did Kaggle actually give us? ───────────────────────────────
# Build 1 spent a session learning that machine_shape is coerced silently: every
# spelling but "NvidiaTeslaT4" quietly became 1x P100, and get_notebook_info kept
# reporting "Gpu" either way. The ONLY proof is what the run itself prints. Same
# discipline here for "TpuV38" — assume nothing, print everything.
import os, sys, time, json, glob, subprocess, platform

print("python      :", sys.version.split()[0])
print("platform    :", platform.platform())
print("cpu count   :", os.cpu_count())
try:
    mem = {l.split(":")[0]: l.split(":")[1].strip()
           for l in open("/proc/meminfo").read().splitlines()[:3]}
    print("host RAM    :", mem)
except Exception as e:
    print("host RAM    : unreadable", e)
for p in ("/kaggle/working", "/kaggle/temp", "/tmp"):
    try:
        s = os.statvfs(p)
        print(f"disk {p:16s}: {s.f_bavail*s.f_frsize/2**30:.1f} GB free")
    except Exception as e:
        print(f"disk {p:16s}: {e}")

print("\n-- TPU-related environment --")
for k in sorted(os.environ):
    if any(t in k.upper() for t in ("TPU", "XLA", "XRT", "PJRT", "LIBTPU")):
        print(f"  {k} = {os.environ[k]}")

print("\n-- accelerator devices on the box --")
for cmd in (["ls", "-l", "/dev/accel0"], ["ls", "/dev/"]):
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=20).stdout
        if cmd[1] == "-l":
            print("  /dev/accel0:", out.strip() or "absent")
        else:
            print("  accel nodes:", [d for d in out.split() if "accel" in d or "vfio" in d] or "none")
    except Exception as e:
        print("  ", cmd, e)

# proj package: %%writefile below keeps model.py visible in the notebook AND importable.
# Without this the writefile lands in a directory that does not exist.
import pathlib
PKG = pathlib.Path("/kaggle/working/proj"); PKG.mkdir(parents=True, exist_ok=True)
(PKG / "__init__.py").touch()
sys.path.insert(0, "/kaggle/working")
print("\nproj package ready at", PKG)


In [ ]:
# ── Cell 2 · torch / torch_xla, and how many cores we really get ─────────────
import torch
print("torch       :", torch.__version__)
print("cuda avail  :", torch.cuda.is_available(), "(expected False on a TPU node)")

try:
    import torch_xla
    print("torch_xla   :", torch_xla.__version__)
except Exception as e:
    print("torch_xla   : IMPORT FAILED ->", type(e).__name__, e)
    raise SystemExit("No torch_xla: this image cannot train PyTorch on TPU. Stop here.")

# PJRT is the runtime torch_xla >= 2.0 uses; XRT is the legacy one. Kaggle's image
# usually sets PJRT_DEVICE itself, but say so out loud either way.
os.environ.setdefault("PJRT_DEVICE", "TPU")
print("PJRT_DEVICE :", os.environ.get("PJRT_DEVICE"))

import torch_xla.core.xla_model as xm
try:
    import torch_xla.runtime as xr
    print("world size  :", xr.world_size(), " global ordinal:", xr.global_ordinal())
    print("device kind :", xr.device_type())
except Exception as e:
    print("torch_xla.runtime unavailable:", e)

dev = xm.xla_device()
print("xla_device  :", dev)
try:
    print("device HW   :", xm.xla_device_hw(dev))
except Exception as e:
    print("device HW   :", e)
try:
    devs = xm.get_xla_supported_devices()
    print("supported   :", devs, f"  -> {len(devs)} core(s) visible to THIS process")
except Exception as e:
    print("supported   :", e)
try:
    print("mem info    :", xm.get_memory_info(dev))
except Exception as e:
    print("mem info    :", e)


In [ ]:
# ── Cell 3 · does bf16 arithmetic actually work, and how fast ────────────────
# TPU has no fp16 path worth using: its native reduced precision is bfloat16, which
# keeps fp32's exponent range. That matters here — the GPU run died of an fp16
# dynamic-range blow-up at round 25, and bf16 is exactly the format that does not
# have that failure mode.
import torch, torch_xla.core.xla_model as xm
dev = xm.xla_device()

for dt in (torch.float32, torch.bfloat16, torch.float16):
    try:
        a = torch.randn(1024, 1024, dtype=dt, device=dev)
        b = torch.randn(1024, 1024, dtype=dt, device=dev)
        c = (a @ b).sum()
        xm.mark_step()
        v = c.cpu().item()
        print(f"  {str(dt):20s} matmul ok, finite={torch.isfinite(torch.tensor(v)).item()}")
    except Exception as e:
        print(f"  {str(dt):20s} FAILED: {type(e).__name__}: {str(e)[:120]}")

# throughput, so the 50-round budget can be estimated from measurement not hope
for dt in (torch.float32, torch.bfloat16):
    a = torch.randn(4096, 4096, dtype=dt, device=dev)
    b = torch.randn(4096, 4096, dtype=dt, device=dev)
    (a @ b).sum(); xm.mark_step()                       # warm the compile cache
    t0 = time.time(); n = 30
    for _ in range(n):
        c = a @ b
    xm.mark_step(); c.cpu()
    dt_s = time.time() - t0
    print(f"  {str(dt):20s} {n} x 4096^3 matmul in {dt_s:.2f}s "
          f"-> {n*2*4096**3/dt_s/1e12:.1f} TFLOP/s")


In [ ]:
%%writefile /kaggle/working/proj/model.py
"""EDL-CMSO feature extractor + DAGSNet. Equation numbers refer to Khan et al. 2025.

Build 2 differs from build 1 in exactly two places, both marked below:
  * Eq. (28) concatenates the RAW wavelet patch, not its embedding  -> fused = P + 2d
  * a CMSO channel mask sits between fusion and DAGSNet             -> the paper's order
"""
import math
import torch, torch.nn as nn, torch.nn.functional as F

# Everything the CMSO probe must reproduce bit-for-bit. The mask is selected on the
# features this half of the network emits at initialisation, so training has to start
# from these exact weights or the mask refers to a projection that no longer exists.
EXTRACTOR_PREFIXES = ("dwt.", "patch.", "vit.", "vit_norm.", "gat.")


class HaarDWT(nn.Module):
    """Eq. (19). Haar (db1), level 1, as a fixed non-learnable stride-2 conv so the
    transform stays on-GPU inside the graph. Returns [C_A || C_D]."""
    def __init__(self):
        super().__init__()
        s = 1.0 / math.sqrt(2.0)
        filt = torch.tensor([[[s, s]], [[-s, s]]])          # (2,1,2): lo=approx, hi=detail
        self.register_buffer("filt", filt)

    def forward(self, x):                                    # (B, n)
        if x.shape[1] % 2:
            x = F.pad(x, (0, 1))
        y = F.conv1d(x.unsqueeze(1), self.filt, stride=2)     # (B, 2, n/2)
        return torch.cat([y[:, 0], y[:, 1]], dim=1)           # (B, n)


class PatchEmbed(nn.Module):
    """Eq. (20)-(21). k patches of length P, linear -> d, plus a learned E_pos.

    Returns (x_patch, z'). x_patch is the RAW wavelet patch and is the X_wavelet term of
    Eq. (28) read literally — build 2's decision C. Build 1 returned the projection z
    instead, making all three fusion terms d-dimensional."""
    def __init__(self, n_in, patch, d):
        super().__init__()
        self.patch = patch
        self.k = math.ceil(n_in / patch)
        self.pad = self.k * patch - n_in
        self.proj = nn.Linear(patch, d)
        self.pos = nn.Parameter(torch.zeros(1, self.k, d))
        nn.init.trunc_normal_(self.pos, std=0.02)

    def forward(self, xw):
        if self.pad:
            xw = F.pad(xw, (0, self.pad))
        xp = xw.view(xw.size(0), self.k, self.patch)              # raw C_A||C_D patches
        return xp, self.proj(xp) + self.pos                       # Eq. (20)-(21)


class ViTBlock(nn.Module):
    """Eq. (22)-(25), pre-norm. nn.MultiheadAttention is exactly Eq. (22)-(24):
    scaled dot-product over Q=z'W_Q, K=z'W_K, V=z'W_V, heads concatenated then W_O."""
    def __init__(self, d, heads, mlp_ratio, drop):
        super().__init__()
        self.n1 = nn.LayerNorm(d)
        self.att = nn.MultiheadAttention(d, heads, dropout=drop, batch_first=True)
        self.n2 = nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, mlp_ratio * d), nn.GELU(),
                                 nn.Dropout(drop), nn.Linear(mlp_ratio * d, d), nn.Dropout(drop))

    def forward(self, z):
        h = self.n1(z)
        z = z + self.att(h, h, h, need_weights=False)[0]      # Eq. (22)-(24)
        return z + self.ffn(self.n2(z))                        # Eq. (25)


class GATLayer(nn.Module):
    """Eq. (26)-(27). The paper never defines N(i); Eq. (26)'s numerator is all-pairs,
    so the graph is fully connected over the k patch nodes.
    a^T[W_h z_i || W_h z_j] = a_src^T W_h z_i + a_dst^T W_h z_j — two matmuls, not k^2
    concatenations."""
    def __init__(self, d_in, d_out, heads, slope, drop):
        super().__init__()
        assert d_out % heads == 0
        self.h, self.dh = heads, d_out // heads
        self.W = nn.Linear(d_in, d_out, bias=False)            # W_h
        self.a_src = nn.Parameter(torch.empty(1, heads, 1, self.dh))
        self.a_dst = nn.Parameter(torch.empty(1, heads, 1, self.dh))
        nn.init.xavier_uniform_(self.a_src); nn.init.xavier_uniform_(self.a_dst)
        self.slope = slope
        self.drop = nn.Dropout(drop)

    def forward(self, z):                                       # (B,k,d_in)
        B, k, _ = z.shape
        h = self.W(z).view(B, k, self.h, self.dh).transpose(1, 2)          # (B,H,k,dh)
        e = (h * self.a_src).sum(-1).unsqueeze(-1) \
          + (h * self.a_dst).sum(-1).unsqueeze(-2)                          # (B,H,k,k)
        alpha = self.drop(torch.softmax(F.leaky_relu(e, self.slope), dim=-1))   # Eq. (26)
        out = (alpha @ h).transpose(1, 2).reshape(B, k, -1)                 # Eq. (27)
        return F.elu(out)                                                    # sigma


# ───────────────────────── DAGSNet backbones (1-D) ─────────────────────────

def cbr(i, o, k):
    return nn.Sequential(nn.Conv1d(i, o, k, padding=k // 2, bias=False),
                         nn.BatchNorm1d(o), nn.ReLU(inplace=True))


class DenseNet1d(nn.Module):
    """Eq. (38)-(39): every layer consumes the concatenation of all preceding maps."""
    def __init__(self, cin, growth, layers):
        super().__init__()
        self.blocks = nn.ModuleList([cbr(cin + i * growth, growth, 3) for i in range(layers)])
        self.out_ch = cin + layers * growth

    def forward(self, x):
        for b in self.blocks:
            x = torch.cat([x, b(x)], dim=1)                     # Eq. (38)
        return x                                                # Eq. (39)


class Inception1d(nn.Module):
    """Eq. (40): parallel 1x1 / 3x3 / 5x5 / pool branches, concatenated."""
    def __init__(self, cin, c):
        super().__init__()
        self.b1 = cbr(cin, c, 1)
        self.b3 = nn.Sequential(cbr(cin, c, 1), cbr(c, c, 3))
        self.b5 = nn.Sequential(cbr(cin, c, 1), cbr(c, c, 5))
        self.bp = nn.Sequential(nn.MaxPool1d(3, 1, 1), cbr(cin, c, 1))
        self.out_ch = 4 * c

    def forward(self, x):
        return torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1)


class GoogleNet1d(nn.Module):
    """Eq. (40)-(41): stacked inception modules."""
    def __init__(self, cin, modules_n, c=32):
        super().__init__()
        mods, ch = [], cin
        for _ in range(modules_n):
            m = Inception1d(ch, c); mods.append(m); ch = m.out_ch
        self.net = nn.Sequential(*mods); self.out_ch = ch

    def forward(self, x):
        return self.net(x)


class AlexNet1d(nn.Module):
    """Eq. (42)-(44): conv+ReLU stack with max pooling. The token axis is only k long,
    so pooling is ceil_mode to avoid collapsing it to zero."""
    def __init__(self, cin, ch=128):
        super().__init__()
        self.net = nn.Sequential(
            cbr(cin, ch, 3), nn.MaxPool1d(2, ceil_mode=True),
            cbr(ch, ch, 3),  nn.MaxPool1d(2, ceil_mode=True),
            cbr(ch, ch, 3))
        self.out_ch = ch

    def forward(self, x):
        return self.net(x)


class Fire1d(nn.Module):
    """Eq. (45)-(46): 1x1 squeeze feeding parallel 1x1 and 3x3 expands."""
    def __init__(self, cin, sq, ex):
        super().__init__()
        self.squeeze = cbr(cin, sq, 1)                          # Eq. (46)
        self.e1 = cbr(sq, ex, 1)
        self.e3 = cbr(sq, ex, 3)
        self.out_ch = 2 * ex

    def forward(self, x):
        s = self.squeeze(x)
        return torch.cat([self.e1(s), self.e3(s)], dim=1)       # Eq. (45)


class SqueezeNet1d(nn.Module):
    def __init__(self, cin, modules_n, sq=32, ex=48):
        super().__init__()
        mods, ch = [], cin
        for _ in range(modules_n):
            m = Fire1d(ch, sq, ex); mods.append(m); ch = m.out_ch
        self.net = nn.Sequential(*mods); self.out_ch = ch

    def forward(self, x):
        return self.net(x)


class EDLCMSO(nn.Module):
    """DWT -> ViT -> GAT -> fusion (Eq. 28) -> CMSO mask (Eq. 29-37) -> DAGSNet (Eq. 47-48).

    n_features is ALWAYS the full input width (66): build 2 selects fused channels, not
    input columns, so nothing upstream of Eq. (28) ever changes shape between runs.
    sel_ch = None builds the unmasked network, which is what the CMSO probe uses."""

    def __init__(self, cfg, n_features, sel_ch=None):
        super().__init__()
        d = cfg["d_model"]
        self.dwt = HaarDWT()
        # HaarDWT pads an odd-length input to even, so the coefficient vector it returns
        # is n + (n % 2) long — not n. PatchEmbed must be sized on THAT.
        dwt_len = n_features + (n_features % 2)
        self.patch = PatchEmbed(dwt_len, cfg["patch_len"], d)
        self.vit = nn.Sequential(*[ViTBlock(d, cfg["vit_heads"], cfg["vit_mlp_ratio"], cfg["dropout"])
                                   for _ in range(cfg["vit_layers"])])
        self.vit_norm = nn.LayerNorm(d)
        self.gat = GATLayer(d, d, cfg["gat_heads"], cfg["gat_slope"], cfg["dropout"])

        # Eq. (28) read literally: raw wavelet patch || z_final || z_new.
        self.fused_dim = cfg["patch_len"] + 2 * d
        if sel_ch is None:
            sel_ch = list(range(self.fused_dim))
        sel = torch.as_tensor(sel_ch, dtype=torch.long)
        assert sel.numel() > 0 and int(sel.max()) < self.fused_dim, "channel mask out of range"
        self.register_buffer("sel_ch", sel)

        s = cfg["stem_ch"]
        self.stems = nn.ModuleList([cbr(sel.numel(), s, 1) for _ in range(4)])
        self.dense  = DenseNet1d(s, cfg["dense_growth"], cfg["dense_layers"])
        self.google = GoogleNet1d(s, cfg["incep_modules"])
        self.alex   = AlexNet1d(s)
        self.squeeze= SqueezeNet1d(s, cfg["fire_modules"])
        comb = self.dense.out_ch + self.google.out_ch + self.alex.out_ch + self.squeeze.out_ch

        self.head = nn.Sequential(                               # Eq. (48)
            nn.LayerNorm(comb), nn.Dropout(cfg["dropout"]),
            nn.Linear(comb, 256), nn.ReLU(inplace=True),
            nn.Dropout(cfg["dropout"]), nn.Linear(256, cfg["num_classes"]))

    # §4.8 in one call — this is the half CMSO sees, and the half init_state pins down.
    def fuse(self, x):
        xw = self.dwt(x)                                         # Eq. (19)
        xp, zp = self.patch(xw)                                  # Eq. (20)-(21)
        z_final = self.vit_norm(self.vit(zp))                    # Eq. (22)-(25)
        z_new = self.gat(z_final)                                # Eq. (26)-(27)
        return torch.cat([xp, z_final, z_new], dim=-1).transpose(1, 2)   # Eq. (28) (B,fused,k)

    def forward(self, x):
        Fm = self.fuse(x).index_select(1, self.sel_ch)           # §4.9 acts here
        feats = [gp(stem(Fm)) for stem, gp in
                 zip(self.stems, [self.dense, self.google, self.alex, self.squeeze])]
        pooled = [f.mean(dim=-1) for f in feats]                 # global average pool
        return self.head(torch.cat(pooled, dim=1))               # Eq. (47) -> (48)


def build_model(cfg, n_features, sel_ch=None):
    return EDLCMSO(cfg, n_features, sel_ch)


def extractor_state(model):
    """Just §4.8's parameters — the ones the CMSO mask was selected against."""
    core = getattr(model, "module", model)
    return {k: v.detach().cpu().clone() for k, v in core.state_dict().items()
            if k.startswith(EXTRACTOR_PREFIXES)}


def load_extractor_state(model, sd):
    """Restore §4.8 exactly. DAGSNet keys are expected to be missing — its width depends
    on |S|, which was not known when the probe was built."""
    core = getattr(model, "module", model)
    own = set(core.state_dict())
    unknown = [k for k in sd if k not in own]
    assert not unknown, f"extractor_init.pt has keys this model does not: {unknown[:5]}"
    missing, unexpected = core.load_state_dict(sd, strict=False)
    assert not unexpected, unexpected
    assert not [k for k in missing if k.startswith(EXTRACTOR_PREFIXES)], \
        "extractor_init.pt did not cover every §4.8 parameter"
    return len(sd)


class SurrogateCNN(nn.Module):
    """Fitness evaluator for CMSO (deviation 19). Not part of the paper's model — it is a
    miniature DAGSNet: 1-D convolutions over the same k token positions, global pool,
    linear head. Structurally analogous to the real classifier so the fitness reflects
    what DAGSNet will see, but small enough that 1000 candidate subsets cost minutes
    rather than the weeks a full-wrapper evaluation would."""
    def __init__(self, cin, n_out, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(cin, hidden, 3, padding=1, bias=False),
            nn.BatchNorm1d(hidden), nn.ReLU(inplace=True),
            nn.Conv1d(hidden, hidden, 3, padding=1, bias=False),
            nn.BatchNorm1d(hidden), nn.ReLU(inplace=True))
        self.head = nn.Linear(hidden, n_out)

    def forward(self, x):                                        # (B, cin, k)
        return self.head(self.net(x).mean(-1))


In [ ]:
# ── Cell 5 · the REAL model, one training step on XLA ────────────────────────
# The whole point of the probe. If EDLCMSO compiles and steps here, the port is a
# plumbing job; if it does not, we find out now for minutes of quota instead of
# discovering it 9 hours into a run.
import torch, torch.nn as nn, time
import torch_xla.core.xla_model as xm
sys.path.insert(0, "/kaggle/working")
from proj.model import build_model

CFG = dict(patch_len=6, d_model=128, vit_layers=2, vit_heads=4, vit_mlp_ratio=2,
           gat_heads=4, gat_slope=0.2, dropout=0.1, stem_ch=96, dense_growth=32,
           dense_layers=3, incep_modules=2, fire_modules=3, num_classes=16)
SEL = list(range(0, 262, 2))            # a 131-channel mask, the size CMSO produced

dev = xm.xla_device()
model = build_model(CFG, 66, SEL).to(dev)
print(f"params: {sum(p.numel() for p in model.parameters()):,}")
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
crit = nn.CrossEntropyLoss()

B = 2048
x = torch.randn(B, 66, device=dev)
y = torch.randint(0, 16, (B,), device=dev)

def step(use_bf16):
    opt.zero_grad(set_to_none=True)
    if use_bf16:
        with torch.autocast("xla", dtype=torch.bfloat16):
            loss = crit(model(x), y)
    else:
        loss = crit(model(x), y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    xm.optimizer_step(opt)              # this is the XLA all-reduce + step
    return loss

for label, bf16 in (("fp32", False), ("bf16-autocast", True)):
    try:
        t0 = time.time(); step(bf16); xm.mark_step()
        l = step(bf16); xm.mark_step(); first = time.time() - t0    # includes compile
        t0 = time.time(); n = 20
        for _ in range(n):
            l = step(bf16)
        xm.mark_step(); v = l.cpu().item()
        per = (time.time() - t0) / n
        print(f"  {label:15s} loss {v:.4f}  {per*1000:7.1f} ms/step at B={B}  "
              f"({B/per:,.0f} samples/s/core, first-step+compile {first:.1f}s)")
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f"  {label:15s} FAILED: {type(e).__name__}")


In [ ]:
# ── Cell 6 · dataset mount, same recursive glob as the GPU notebook ──────────
print("-- /kaggle/input --")
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.count("/") - 2
        if depth > 3:
            dirs[:] = []; continue
        print("  " * depth + os.path.basename(root) + "/" +
              (f"   [{len(files)} files]" if files else ""))
else:
    print("  /kaggle/input does not exist")

cands = sorted(p for p in glob.glob("/kaggle/input/**/train", recursive=True)
               if os.path.isdir(p) and glob.glob(os.path.join(p, "*.parquet")))
print("\ntrain dirs with parquet:", cands or "NONE — dataset not attached")
if cands:
    import pyarrow.parquet as pq
    d = os.path.dirname(cands[0])
    tr = sorted(glob.glob(os.path.join(d, "train", "*.parquet")))
    te = sorted(glob.glob(os.path.join(d, "test", "*.parquet")))
    print(f"train shards {len(tr)}   test shards {len(te)}")
    t0 = time.time()
    n = sum(pq.ParquetFile(f).metadata.num_rows for f in tr)
    print(f"train rows from footers: {n:,}  ({time.time()-t0:.1f}s)")
    print("pyarrow available:", pq.__name__)

print("\n" + "="*70)
print("PROBE COMPLETE — read the numbers above before writing the TPU notebook.")
print("="*70)
